In [6]:
#TODO Agregar el chequeo de que se escoja la edge mas barata entre la misma ruta de dos mismas ciudades
#TODO Agregar father para el mejor camino
#TODO Hacer las transbordaciones entre lineas que ahi si es donde se agrega los 4 minutos

import numpy as np
import sys

class Edge:
    def __init__(self,
                 from_stop_id,
                 from_stop_name,
                 to_stop_id,
                 to_stop_name,
                 route_id,
                 route_short_name,
                 route_long_name,
                 mean_travel_time_min
                 ):
        self.from_stop_id = from_stop_id
        self.from_stop_name = from_stop_name
        self.to_stop_id = to_stop_id
        self.to_stop_name = to_stop_name
        self.route_id = route_id
        self.route_short_name = route_short_name
        self.route_long_name = route_long_name
        self.mean_travel_time_min = mean_travel_time_min
        self.mean_travel_time_min *=60





class Graph:
    def __init__(self, matrix, edges, cities, id_to_idx, idx_to_id):
        self.original_matrix = np.copy(matrix)
        self.matrix = matrix
        self.edges = edges
        self.cities = cities
        self.id_to_idx = id_to_idx
        self.idx_to_id = idx_to_id
        self.parent = np.full((len(cities), len(cities)), -1, dtype = int)
        self.build_parent()
        self.floyd_war()





    def floyd_war(self):
        n = len(self.cities)
        for k in range(n):
            for fr in range(n):
                for tt in range(n):
                    if self.matrix[fr][tt] > self.matrix[fr][k] + self.matrix[k][tt]:
                        self.matrix[fr][tt] = min(self.matrix[fr][tt], self.matrix[fr][k] + self.matrix[k][tt])
                        self.parent[fr][tt] = self.parent[k][tt]


    def build_parent(self):
        n = len(self.cities)
        for i in range(n):
            for j in range(n):
                if i != j and self.matrix[i][j] != np.inf:
                    self.parent[i][j] = i

    def get_path_cost(self, A_, B_):
        A = self.id_to_idx[A_]
        B = self.id_to_idx[B_]

        if self.matrix[A][B] == np.inf or A == B:
            return {'p': '-1', 'c': -1}
        path_costo = {}
        path = []

        current = B

        path.append(current)


        while(current != A):
            current = self.parent[A][current]
            path.append(current)


        path.reverse()
        for i in range(len(path)):
           path[i] = self.idx_to_id[path[i]]

        path_costo['p'] = path
        path_costo['c'] = self.matrix[A][B]
        return path_costo


#la matriz de adjacencia
adj_ma = {}
#las edges para guardar toda info
edges = {}
#guardar el numero unico de ciudades
cities = set()
#mapea nombre a un indice
id_to_idx = {}

idx_to_id = {}

#Guarda ciudad: lineas
city_line = {}

TRANSBORDO_MIN = 4   # minutos que cuesta cambiar de linea


def agregar_transbordos(stop_id, stop_name, route_id):
    # se busca por nombre porque los ids no siempre coinciden (SALTODELAGUA vs SALTOAGUA)
    nombre = ' '.join(stop_name.split())
    if nombre not in city_line:
        city_line[nombre] = {}

    # si ya vi esta estacion en esta linea, ya tiene sus transbordos
    if route_id in city_line[nombre]:
        return

    # conectar con las otras lineas donde esta estacion ya existia (ida y vuelta)
    for otra_linea, otro_id in city_line[nombre].items():
        a = id_to_idx[stop_id]
        b = id_to_idx[otro_id]
        for fr, tt in ((a, b), (b, a)):
            if fr not in adj_ma:
                adj_ma[fr] = []
            adj_ma[fr].append(tt)
            edges[(fr, tt)] = Edge(idx_to_id[fr], nombre, idx_to_id[tt], nombre,
                                   "TRANSBORDO", "T", "Transbordo", TRANSBORDO_MIN)

    city_line[nombre][route_id] = stop_id


with open("/content/metro_cdmx_aristas.csv", mode = 'r') as file:
    i = 0
    flag = False
    for line in file:
        if flag == False:
            flag = True
            continue
        e = line.strip().split(',')
        cities.add(e[0])
        cities.add(e[2])

        # Actualizar el mapa para que cada nombre este mapeado a un numero
        if id_to_idx.get(e[0], -1) == -1:
            id_to_idx[e[0]] = i
            idx_to_id[i] = e[0]
            i+=1

        if id_to_idx.get(e[2], -1) == -1:
            id_to_idx[e[2]] = i
            idx_to_id[i] = e[2]
            i+=1


        f = id_to_idx[e[0]]
        to = id_to_idx[e[2]]

        if f not in adj_ma:
            adj_ma[f] = []

        adj_ma[f].append(to)
        #--------------------------------Aca el chequeo---------------------------
        if edges.get((f, to), -1) != -1:
            if edges[(f, to)].mean_travel_time_min > float(e[7]) * 60:
                edges[(f,to)] = Edge(e[0], e[1], e[2], e[3], e[4], e[5], e[6], float(e[7]))
        else:
            edges[(f, to)] = Edge(e[0], e[1], e[2], e[3], e[4], e[5], e[6], float(e[7]))

        #----------------------------aca se agrega el edge del transbordo-
        agregar_transbordos(e[0], e[1], e[4])
        agregar_transbordos(e[2], e[3], e[4])
file.close()

matrix = np.full((len(cities), len(cities)), np.inf, dtype = float)
np.fill_diagonal(matrix, 0)

for city in cities:
    fr = id_to_idx[city]
    if fr not in adj_ma:
        continue
    for tt in adj_ma[fr]:
        matrix[fr][tt] = edges[(fr,tt)].mean_travel_time_min


g = Graph(matrix, edges, cities, id_to_idx, idx_to_id)

# ------------------------------ Interfaz ------------------------------
import ipywidgets as widgets
from IPython.display import display

# nombre de cada estacion, para mostrarlo en los menus
nombres = {}
for ed in edges.values():
    nombres[ed.from_stop_id] = ed.from_stop_name
    nombres[ed.to_stop_id] = ed.to_stop_name

# (lo que se ve, valor que se guarda)
opciones = sorted([(f"{nombres[i]} ({i})", i) for i in id_to_idx])

origen = widgets.Dropdown(options=opciones, description="Origen:", layout=widgets.Layout(width="500px"))
destino = widgets.Dropdown(options=opciones, description="Destino:", layout=widgets.Layout(width="500px"))
boton = widgets.Button(description="Buscar camino", button_style="primary")
salida = widgets.Output()


def buscar(_):
    with salida:
        salida.clear_output()
        result = g.get_path_cost(origen.value, destino.value)

        if result['p'] == '-1':
            print("No hay camino o el origen y el destino son iguales")
            return

        total = result['c']
        minutos = int(total // 60)
        segundos = int(total % 60)

        print(f"Tiempo total (min:seg): {minutos}:{segundos:02d}")
        print(f"Estaciones: {len(result['p'])}\n")
        for n, sid in enumerate(result['p'], start=1):
            print(f"{n}. {nombres[sid]} ({sid})")


boton.on_click(buscar)
display(widgets.VBox([origen, destino, boton, salida]))

Tiempo total (min:seg): 15:39
Estaciones: 10

1. Balbuena (0200L1-BALBUENA)
2. Moctezuma (0200L1_MOCTEZUMA)
3. San Lázaro (0200L1-SANLAZARO)
4. Candelaria (0200L1-CANDELARIA)
5. Merced (0200L1-MERCED)
6. Pino Suárez (0200L1-PINOSUAREZ)
7. Isabel la Católica (0200L1-ISABELLACA)
8. Salto del Agua (0200L1-SALTODELAGUA)
9. Balderas (0200L1-BALDERAS)
10. Balderas (0200L3-BALDERAS)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')